# 05 Thesis Figures
Builds the extra result figures for Chapters 4 and 5 from the saved results of Notebooks 02, 02b and 03.
It reads only files in `Results/` (no retraining) and saves PNGs in `Figures/`.

**At a glance**

| | |
|---|---|
| Run on | Laptop. Run it in Windows Jupyter so the figures use Times New Roman |
| Run time | Under a minute |
| Reads | Only saved files in `../Results/` (no training, no raw data) |
| Writes | Seven figures in `../Figures/` (latent search, seeds, thresholds, validation vs test, beta effect, sensitivity, top error features) |
| Run order | Step 6 of 6, after Notebooks 03 and 04. |

In [ ]:
import os, json, glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

BASE = os.path.abspath("..")
RES = os.path.join(BASE, "Results")
TAB = os.path.join(RES, "tables")
FIG = os.path.join(BASE, "Figures")
os.makedirs(FIG, exist_ok=True)

plt.rcParams.update({"figure.dpi": 110, "savefig.dpi": 300, "font.size": 12, "axes.titlesize": 13,
                     "axes.labelsize": 12, "font.family": "serif",
                     "font.serif": ["Times New Roman", "Liberation Serif", "DejaVu Serif"]})
sns.set_style("whitegrid")
COL = {"AE": "#4C72B0", "VAE": "#DD8452", "PCA": "#55A868", "IForest": "#8172B3"}
NAMES = {"ieee": "IEEE-CIS", "baf": "BAF"}

def savefig(name):
    plt.savefig(os.path.join(FIG, name), bbox_inches="tight"); print("saved", name)

runs = pd.read_csv(os.path.join(TAB, "all_runs_test_metrics.csv"), index_col=0)
print(runs.shape)

## Figure: latent size search (Stage A, seed 0, validation period)

In [ ]:
tune = pd.read_csv(os.path.join(RES, "tuning_seed0.csv"))
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, ds in zip(axes, ["ieee", "baf"]):
    for m in ["AE", "VAE"]:
        d = tune[(tune.dataset == ds) & (tune.model == m)].sort_values("latent")
        ax.plot(d.latent, d.val_auc_pr, marker="o", color=COL[m], label=m)
        best = d.loc[d.val_auc_pr.idxmax()]
        ax.scatter([best.latent], [best.val_auc_pr], s=160, facecolors="none", edgecolors=COL[m], linewidths=2)
    ax.set_xticks([8, 16, 32]); ax.set_xlabel("Latent size"); ax.set_ylabel("Validation AUC-PR")
    ax.set_title(NAMES[ds]); ax.legend()
plt.tight_layout(); savefig("latent_size_search.png"); plt.show()

## Figure: AE versus VAE across the five seeds (test period)

In [ ]:
thr = runs[runs.threshold_rule == "calibrated_F1"]
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for col, ds in enumerate(["IEEE-CIS", "BAF"]):
    for row, met in enumerate(["auc_roc", "auc_pr"]):
        ax = axes[row, col]
        d = thr[thr.dataset == ds]
        ae = d[d.model == "AE"].sort_values("seed"); vae = d[d.model == "VAE"].sort_values("seed")
        for (_, a), (_, v) in zip(ae.iterrows(), vae.iterrows()):
            ax.plot([0, 1], [a[met], v[met]], color="grey", lw=0.8, zorder=1)
        ax.scatter(np.zeros(len(ae)), ae[met], color=COL["AE"], s=50, zorder=2, label="AE (one dot per seed)")
        ax.scatter(np.ones(len(vae)), vae[met], color=COL["VAE"], s=50, zorder=2, label="VAE (one dot per seed)")
        for m, ls in [("PCA", "--"), ("IForest", ":")]:
            ax.axhline(d[d.model == m][met].mean(), color=COL[m], ls=ls, lw=1.5,
                       label=("Isolation Forest" if m == "IForest" else m) + " (mean)")
        ax.set_xticks([0, 1]); ax.set_xticklabels(["AE", "VAE"]); ax.set_xlim(-0.5, 1.5)
        ax.set_ylabel("Test " + ("AUC-ROC" if met == "auc_roc" else "AUC-PR")); ax.set_title(ds)
h, l = axes[0, 0].get_legend_handles_labels()
fig.legend(h, l, loc="lower center", ncol=4, fontsize=10, frameon=False)
plt.tight_layout(rect=(0, 0.05, 1, 1)); savefig("ae_vs_vae_seeds.png"); plt.show()

## Figure: the three threshold rules (test period, AE and VAE)

In [ ]:
rule_name = {"fixed_p95": "Fixed 95th", "fixed_p99": "Fixed 99th", "calibrated_F1": "Calibrated"}
d = runs[runs.model.isin(["AE", "VAE"])].copy(); d["rule"] = d.threshold_rule.map(rule_name)
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5))
for col, ds in enumerate(["IEEE-CIS", "BAF"]):
    for row, (met, lab) in enumerate([("f1", "Test F1"), ("flagged_%", "Test records flagged (%)")]):
        ax = axes[row, col]
        sns.barplot(data=d[d.dataset == ds], x="rule", y=met, hue="model", palette=COL, errorbar="sd",
                    order=list(rule_name.values()), ax=ax, capsize=0.1)
        ax.set_xlabel(""); ax.set_ylabel(lab); ax.set_title(ds); ax.legend(title="")
plt.tight_layout(); savefig("threshold_rules.png"); plt.show()

## Figure: validation period versus test period (AUC-PR, final latent size, 5 seeds)

In [ ]:
best = json.load(open(os.path.join(RES, "best_latent.json")))
rows = []
for ds in ["ieee", "baf"]:
    for m in ["AE", "VAE"]:
        for meta in glob.glob(os.path.join(RES, "runs_final", f"{ds}_{m}*_z{best[f'{ds}_{m}']}_s*", "meta.json")):
            j = json.load(open(meta))
            t = runs[(runs.dataset == NAMES[ds]) & (runs.model == m) & (runs.seed == j["seed"]) &
                     (runs.threshold_rule == "calibrated_F1")].auc_pr.iloc[0]
            rows.append({"dataset": NAMES[ds], "model": m, "seed": j["seed"], "Validation": j["val_auc_pr"], "Test": t})
vt = pd.DataFrame(rows)
print(vt.groupby(["dataset", "model"])[["Validation", "Test"]].mean().round(4))
long = vt.melt(id_vars=["dataset", "model", "seed"], var_name="Period", value_name="AUC-PR")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, ds in zip(axes, ["IEEE-CIS", "BAF"]):
    sns.barplot(data=long[long.dataset == ds], x="model", y="AUC-PR", hue="Period", errorbar="sd",
                palette={"Validation": "#9DB4D6", "Test": "#2F5597"}, capsize=0.1, ax=ax)
    ax.set_xlabel(""); ax.set_title(ds); ax.legend(title="")
plt.tight_layout(); savefig("validation_vs_test.png"); plt.show()

## Figure: KL weight, active units and detection (IEEE-CIS VAE)

In [ ]:
sens = pd.read_csv(os.path.join(TAB, "sensitivity_all_runs.csv"))
abl = pd.read_csv(os.path.join(TAB, "table_ablation_beta.csv"), header=[0, 1], index_col=[0, 1, 2])
pts = []
for v, b in [("beta 0.001", 0.001), ("main (dropout 0.2, lr 0.001, beta 0.01)", 0.01), ("beta 0.1", 0.1)]:
    s = sens[(sens.model == "VAE") & (sens.variant == v)]
    pts.append({"beta": b, "auc_pr": s.auc_pr.mean(), "sd": s.auc_pr.std(), "active": s.active_units.mean()})
b1 = abl.loc[("IEEE-CIS", "VAE beta=1", 32)]
pts.append({"beta": 1.0, "auc_pr": b1[("auc_pr", "mean")], "sd": b1[("auc_pr", "std")], "active": b1[("active_units", "mean")]})
p = pd.DataFrame(pts); print(p)
fig, ax1 = plt.subplots(figsize=(7.5, 4.2))
ax1.errorbar(p.beta, p.auc_pr, yerr=p.sd, marker="o", color=COL["VAE"], capsize=4, label="Test AUC-PR")
ae_main = sens[(sens.model == "AE") & (sens.variant.str.startswith("main"))].auc_pr.mean()
ax1.axhline(ae_main, color=COL["AE"], ls="--", label="AE, main settings")
ax1.set_xscale("log"); ax1.set_xlabel("KL weight β (log scale)"); ax1.set_ylabel("Test AUC-PR")
ax2 = ax1.twinx(); ax2.bar(p.beta, p.active, width=p.beta * 0.35, alpha=0.25, color="grey", label="Active latent units")
ax2.set_ylabel("Active latent units"); ax2.grid(False)
h1, l1 = ax1.get_legend_handles_labels(); h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="lower left", fontsize=9)
ax1.set_title("IEEE-CIS VAE: β = 0.001, 0.01, 0.1 (seeds 0 to 2) and β = 1 (5 seeds)")
plt.tight_layout(); savefig("beta_effect.png"); plt.show()

## Figure: sensitivity check, AE versus VAE in each matched variant (IEEE-CIS, seeds 0 to 2)

In [ ]:
lab = {"main (dropout 0.2, lr 0.001, beta 0.01)": "Main settings", "no dropout": "No dropout",
       "learning rate 0.0003": "Learning rate 0.0003"}
s = sens[sens.variant.isin(lab)].copy(); s["Variant"] = s.variant.map(lab)
fig, ax = plt.subplots(figsize=(8, 4.2))
sns.barplot(data=s, x="Variant", y="auc_pr", hue="model", palette=COL, errorbar="sd", capsize=0.1,
            order=list(lab.values()), ax=ax)
ifo = runs[(runs.dataset == "IEEE-CIS") & (runs.model == "IForest") & (runs.threshold_rule == "calibrated_F1")].auc_pr.mean()
ax.axhline(ifo, color=COL["IForest"], ls=":", lw=1.5, label="Isolation Forest (5 seeds)")
ax.set_xlabel(""); ax.set_ylabel("Test AUC-PR"); ax.legend(title="", fontsize=9)
plt.tight_layout(); savefig("sensitivity_check.png"); plt.show()

## Figure: the ten features with the largest share of reconstruction error (AE, test period)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
for ax, ds in zip(axes, ["ieee", "baf"]):
    t = pd.read_csv(os.path.join(TAB, f"table_feature_errors_{ds}_AE.csv")).sort_values("error_share_%", ascending=False).head(10)[::-1]
    cmap = plt.cm.RdYlGn
    colors = [cmap((a - 0.5) / 0.25) for a in t.univariate_auc]
    ax.barh(t.feature, t["error_share_%"], color=colors, edgecolor="grey")
    for y, (v, a) in enumerate(zip(t["error_share_%"], t.univariate_auc)):
        ax.text(v + 0.3, y, f"AUC {a:.2f}", va="center", fontsize=9)
    ax.set_xlabel("Share of total test reconstruction error (%)"); ax.set_title(NAMES[ds])
    ax.set_xlim(0, t["error_share_%"].max() * 1.3)
plt.suptitle("Bar colour: fraud signal of the feature on its own (univariate AUC, red = weak, green = strong)", fontsize=10, y=0.02)
plt.tight_layout(rect=(0, 0.04, 1, 1)); savefig("top_error_features.png"); plt.show()